<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-01-attention-by-hand-and-by-code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 (graded) — Attention, by hand and by code
**Course 2: Generative AI and LLMs with Python — Chapter 1: On-ramp: neurons → the transformer**

**What you'll submit:** working scaled dot-product attention and multi-head attention
(passing the test harness below), an attention-map visualization on a toy sentence, and a
short explainer of what each head appears to have learned.

## 1. Scaled dot-product attention
Fill in the `TODO`.

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K, V: (batch, heads, seq, d_k). Returns (output, attn_weights)."""
    d_k = K.shape[-1]
    # TODO: scores = Q @ K^T / sqrt(d_k)
    scores = None
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    # TODO: weights = softmax(scores, dim=-1)
    weights = None
    # TODO: output = weights @ V
    output = None
    return output, weights

## 2. Test harness — run this before continuing

In [ ]:
def test_scaled_dot_product_attention():
    B, H, S, D = 2, 3, 5, 8
    Q = torch.randn(B, H, S, D)
    K = torch.randn(B, H, S, D)
    V = torch.randn(B, H, S, D)

    out, weights = scaled_dot_product_attention(Q, K, V)
    assert out is not None, 'output is None — implement the TODOs above'
    assert out.shape == (B, H, S, D), f'expected output shape {(B, H, S, D)}, got {out.shape}'
    assert weights.shape == (B, H, S, S), f'expected weights shape {(B, H, S, S)}, got {weights.shape}'
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), \
        'attention weights should sum to 1 along the last dim (softmax not applied correctly)'

    # a causal mask should make position 0 attend ONLY to itself
    causal_mask = torch.tril(torch.ones(S, S)).view(1, 1, S, S)
    _, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
    assert torch.allclose(masked_weights[..., 0, 1:], torch.zeros_like(masked_weights[..., 0, 1:]), atol=1e-5), \
        'with a causal mask, position 0 should have zero weight on all future positions'

    print('All scaled_dot_product_attention tests PASSED.')

test_scaled_dot_product_attention()

## 3. Multi-head attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # x: (batch, seq, d_model) -> (batch, heads, seq, d_k)
        B, S, _ = x.shape
        # TODO: reshape to (B, S, n_heads, d_k) then transpose to (B, n_heads, S, d_k)
        return None

    def forward(self, x, mask=None):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        out, weights = scaled_dot_product_attention(Q, K, V, mask)
        # TODO: merge heads back: (B, n_heads, S, d_k) -> (B, S, d_model)
        B, H, S, D = out.shape
        merged = None
        return self.W_o(merged), weights


def test_multi_head_attention():
    mha = MultiHeadAttention(d_model=32, n_heads=4)
    x = torch.randn(2, 6, 32)
    out, weights = mha(x)
    assert out is not None and out.shape == (2, 6, 32), f'expected (2, 6, 32), got {getattr(out, "shape", None)}'
    assert weights.shape == (2, 4, 6, 6)
    print('MultiHeadAttention test PASSED.')

test_multi_head_attention()

## 4. Visualize attention on a toy sentence

In [ ]:
import matplotlib.pyplot as plt

tokens = ['the', 'bank', 'raised', 'interest', 'rates', 'again']
d_model, n_heads = 32, 4
torch.manual_seed(1)
embed = nn.Embedding(len(tokens), d_model)
x = embed(torch.arange(len(tokens))).unsqueeze(0)  # (1, seq, d_model) — untrained, illustrative only

mha = MultiHeadAttention(d_model, n_heads)
_, attn_weights = mha(x)  # (1, heads, seq, seq)

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
for h in range(n_heads):
    ax = axes[h]
    ax.imshow(attn_weights[0, h].detach().numpy(), cmap='viridis')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
    ax.set_title(f'head {h}')
plt.suptitle('Attention weights per head (untrained weights — pattern is illustrative, not learned)')
plt.tight_layout()
plt.show()

## 5. Explainer (fill in)
With random/untrained weights, the attention pattern above is not meaningful yet — that's
expected and worth explaining in your own words: what *would* a trained head's pattern show
for a sentence like this, and why can't you read anything into an untrained one?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 1: On-ramp*